In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, roc_auc_score

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier

In [9]:
df = pd.read_csv("../data/train_feature_engineered.csv")

print(df.shape)
df.head()

(221, 22)


,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,log1p_area_first,log1p_growth,radial_growth_m,spread_bearing_deg,spread_bearing_sin,spread_bearing_cos,...,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,3,4.265188,0,4.390693,4.390693,1.354787,2.303303,70.130507,0.940469,0.339879,...,7.275611e-02,0.886373,-0.054649,0.054649,-1.937219,19,4,5,18.892512,0
1,2,1.169918,0,2.297246,2.297246,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000e+00,0.000000,-0.568898,0.568898,-0.000000,4,4,6,22.048108,1
2,4,4.777526,0,4.677329,4.677329,0.000000,0.000000,0.000000,0.000000,1.000000,...,7.965118e-14,0.000000,0.882385,0.882385,0.000000,22,4,8,0.888895,1
3,1,0.000000,1,4.228746,4.228746,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,2,4.975273,0,3.600946,3.600946,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000e+00,0.000000,0.934634,0.934634,-0.000000,21,5,7,44.990274,0


In [10]:
X = df.drop(columns=["time_to_hit_hours","event"])
y_cls = df["event"]

In [11]:
X_train, X_test, y_cls_train, y_cls_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42
)

In [12]:
cls_models = {
    
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        eval_metric="logloss"
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05
    ),
    
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        verbose=False
    )
}

In [13]:
cls_results = []

for name, model in cls_models.items():
    
    pipe = Pipeline([
        ("model", model)
    ])
    
    pipe.fit(X_train, y_cls_train)
    
    prob = pipe.predict_proba(X_test)[:,1]
    
    auc = roc_auc_score(y_cls_test, prob)
    
    cls_results.append((name, auc))

cls_results = pd.DataFrame(cls_results, columns=["Model","ROC_AUC"])

cls_results.sort_values("ROC_AUC", ascending=False)

[LightGBM] [Info] Number of positive: 51, number of negative: 125
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001054 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 290
[LightGBM] [Info] Number of data points in the train set: 176, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.289773 -> initscore=-0.896488
[LightGBM] [Info] Start training from score -0.896488
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,Model,ROC_AUC
0,RandomForest,1.0
1,XGBoost,1.0
2,LightGBM,1.0
3,CatBoost,1.0
